# Build a GNN Model to seed Nodes

In [71]:
import sys, os
sys.path.append(os.path.abspath("../../"))

In [80]:
os.getcwd()


'/Users/graceegeorge/ML3 Project/contagion-seeding-in-meetup'

In [73]:
%%capture
%run src/training_v2.py

FileNotFoundError: [Errno 2] No such file or directory: '/Users/graceegeorge/ML3 Project/contagion-seeding-in-meetup/contagion-seeding-in-meetup/notebooks/data/G_bipartite.graphml'

In [65]:
print(edge_index_1_torch.shape)
print(edge_attr_1_torch.T.shape)
print(edge_index_2_torch.shape)
print(edge_attr_2_torch.T.shape)

torch.Size([2, 38518])
torch.Size([1, 38518])
torch.Size([2, 257808])
torch.Size([1, 257808])


## Importing 

In [66]:
# General
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder

# Data download
import kagglehub
import shutil

# Plotting 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# Specific to network purposes
import networkx as nx

# PyToch
import torch
import torch_geometric
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

from src.training_v2 import load_and_prepare_training_data
train_dataloader, val_dataloader, static_graph = load_and_prepare_training_data()


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 2 but got size 38518 for tensor number 1 in the list.

In [ ]:
# Adapted code from 
    # https://towardsdatascience.com/graph-attention-networks-in-python-975736ac5c0c/
    # https://www.github.com/diegoantognini/pyGAT/blob/master/
class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout, heads=8):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1)
        self.optmizer = torch.optim.Adam(self.parameters(), lr=0.001)

    def forward(self,x,edge_index):
        h = F.dropout(x, self.dropout, p=0.5, training=self.training)
        h = self.gat1(x,edge_index)
        h = F.elu(h)
        h = F.dropout(h, self.dropout, p=0.5, training=self.training)
        h = self.gat2(h,edge_index)
        return h, F.log_softmax(h, dim=1)
    
    
    
    
    

In [ ]:
model = GAT(
    in_dim = 9,
    hidden_dim = 128,
    dropout=0.5,
    out_dim = 1
)


In [41]:
config = {
    'lr': 1e-3,
    'epochs': 10,
    'project_name': 'meetup-contagion-seeding',
    'run_name': 'mcmc-vectorized-run'
}

trainer = ImitationTrainer(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    static_graph=static_graph,
    config=config,
    use_wandb=True # Switch to False if you dont have wandb setup
)
print("\nStarting Training...")
trainer.train()
trainer.plot_losses()

NameError: name 'train_dataloader' is not defined